[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jingjieyeo/abm_on_colab/blob/main/bacterial_growth_division.ipynb)

# Bacterial Growth and Division Model
## How resource competition influence biofilm dynamics


### Introduction

This Google Colab notebook explores **bacterial population dynamics** through agent-based modeling, building upon movement and navigation concepts from previous exercises. We implement bacterial agents that:

- **Consume environmental resources** to accumulate biomass
- **Undergo cell division** when reaching critical thresholds
- **Compete for limited resources** creating spatial heterogeneity
- **Exhibit emergent population dynamics** from individual behaviors

### Learning Objectives

By the end of this notebook, you will:

1. Understand how individual bacterial behaviors create population-level patterns
2. Explore the difference between exponential and logistic growth
3. Investigate resource competition and carrying capacity
4. Analyze spatial heterogeneity in microbial communities
5. Visualize complete simulation trajectories

## Setup and Installation

First, let's install the Mesa framework and other libraries we'll need.

In [1]:
# Install required packages for Google Colab
!pip install mesa matplotlib numpy pandas seaborn ipywidgets

# Import all required libraries
import mesa
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import pandas as pd
import random
import math
import seaborn as sns

from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector
from IPython.display import clear_output, HTML, display
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, VBox, HBox

import warnings
warnings.filterwarnings('ignore')

print("Mesa version:", mesa.__version__)
print("All libraries imported successfully!")

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.0/197.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.9 MB/s eta 0:00:00
Mesa version: 3.2.0
All libraries imported successfully!


In [2]:
# =============================================================================
# PART 1: RESOURCE ENVIRONMENT
# =============================================================================

class ResourceEnvironment:
    """
    Manages the spatial distribution and dynamics of nutrients in the environment.
    Resources are depleted by bacterial consumption and can regenerate over time.
    """

    def __init__(self, width, height, initial_resource_level=100.0,
                 regeneration_rate=0.1, diffusion_rate=0.05):
        """
        Initialize the resource environment.

        Args:
            width: Grid width
            height: Grid height
            initial_resource_level: Starting resource concentration per cell
            regeneration_rate: Rate at which resources regenerate per step
            diffusion_rate: Rate at which resources diffuse between cells
        """
        self.width = width
        self.height = height
        self.regeneration_rate = regeneration_rate
        self.diffusion_rate = diffusion_rate

        # Initialize resource grid with uniform distribution
        self.resources = np.full((width, height), initial_resource_level, dtype=float)
        self.max_resources = initial_resource_level

    def get_resource_at(self, x, y):
        """Get resource level at specific coordinates."""
        return self.resources[x, y]

    def consume_resource(self, x, y, amount):
        """
        Consume resources at a location.
        Returns the actual amount consumed (limited by availability).
        """
        available = self.resources[x, y]
        consumed = min(amount, available)
        self.resources[x, y] -= consumed
        return consumed

    def regenerate_resources(self):
        """Regenerate resources across the grid."""
        # Add regeneration, but don't exceed maximum
        self.resources = np.minimum(
            self.resources + self.regeneration_rate,
            self.max_resources
        )

    def diffuse_resources(self):
        """Implement simple diffusion of resources between neighboring cells."""
        if self.diffusion_rate <= 0:
            return

        # Create a copy for calculations
        new_resources = self.resources.copy()

        for x in range(self.width):
            for y in range(self.height):
                # Get neighbors (with boundary conditions)
                neighbors = []
                for dx in [-1, 0, 1]:
                    for dy in [-1, 0, 1]:
                        if dx == 0 and dy == 0:
                            continue
                        nx, ny = (x + dx) % self.width, (y + dy) % self.height
                        neighbors.append(self.resources[nx, ny])

                # Average with neighbors
                avg_neighbor = np.mean(neighbors)
                current = self.resources[x, y]

                # Move toward equilibrium
                diff = (avg_neighbor - current) * self.diffusion_rate / 8
                new_resources[x, y] = current + diff

        self.resources = new_resources

    def step(self):
        """Update resources for one time step."""
        self.regenerate_resources()
        self.diffuse_resources()

In [3]:
class BacteriumAgent(Agent):
    """
    A bacterial agent that can:
    - Move randomly or toward resources
    - Consume environmental resources
    - Accumulate biomass over time
    - Divide when reaching critical biomass threshold
    - Die when resources are too scarce
    """

    def __init__(self, model, initial_biomass=1.0, max_biomass=2.0,
                 consumption_rate=0.5, metabolic_cost=0.1, mutation_rate=0.01):
        """
        Initialize a bacterial agent.

        Args:
            model: Reference to the model
            initial_biomass: Starting biomass
            max_biomass: Biomass threshold for division
            consumption_rate: Rate of resource consumption
            metabolic_cost: Energy cost of maintenance per step
            mutation_rate: Probability of parameter mutation during division
        """
        super().__init__(model)

        # Core biological parameters
        self.biomass = initial_biomass
        self.max_biomass = max_biomass
        self.consumption_rate = consumption_rate
        self.metabolic_cost = metabolic_cost
        self.mutation_rate = mutation_rate

        # State tracking
        self.age = 0
        self.generation = 0
        self.energy = initial_biomass
        self.growth_rate = 0.0
        self.is_dividing = False

        # Movement parameters
        self.speed = 1
        self.chemotaxis_strength = 0.3

        # Tracking for analysis
        self.total_consumed = 0.0
        self.division_count = 0

    def sense_resources(self):
        """
        Sense resource availability in the local neighborhood.
        Returns the direction toward highest resource concentration.
        """
        x, y = self.pos
        max_resources = 0
        best_direction = None

        # Check all neighboring cells
        neighbors = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False, radius=1
        )

        for neighbor_pos in neighbors:
            nx, ny = neighbor_pos
            resources = self.model.environment.get_resource_at(nx, ny)
            if resources > max_resources:
                max_resources = resources
                best_direction = neighbor_pos

        return best_direction, max_resources

    def move(self):
        """Move the bacterium using chemotaxis toward resources."""
        # Sense environment for resource gradients
        best_pos, max_resources = self.sense_resources()

        # Decide between random movement and chemotaxis
        if best_pos and random.random() < self.chemotaxis_strength:
            # Move toward resources
            possible_steps = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=False, radius=self.speed
            )

            # Filter to moves that get closer to the resource
            current_x, current_y = self.pos
            target_x, target_y = best_pos

            better_moves = []
            for step_pos in possible_steps:
                step_x, step_y = step_pos
                current_dist = math.sqrt((current_x - target_x)**2 + (current_y - target_y)**2)
                step_dist = math.sqrt((step_x - target_x)**2 + (step_y - target_y)**2)
                if step_dist <= current_dist:
                    better_moves.append(step_pos)

            if better_moves:
                new_position = self.random.choice(better_moves)
            else:
                # Random movement if no good chemotactic move
                possible_steps = self.model.grid.get_neighborhood(
                    self.pos, moore=True, include_center=False, radius=self.speed
                )
                new_position = self.random.choice(possible_steps)
        else:
            # Random movement
            possible_steps = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=False, radius=self.speed
            )
            new_position = self.random.choice(possible_steps)

        # Move to new position
        self.model.grid.move_agent(self, new_position)

    def consume_and_grow(self):
        """
        Consume resources at current location and convert to biomass.
        Implements Monod kinetics for resource consumption.
        """
        x, y = self.pos

        # Get available resources
        available_resources = self.model.environment.get_resource_at(x, y)

        # Calculate consumption using Monod kinetics
        # consumption = max_rate * resources / (half_saturation + resources)
        half_saturation = 10.0  # Half-saturation constant
        max_consumption = self.consumption_rate * self.biomass

        actual_consumption_rate = (max_consumption * available_resources /
                                 (half_saturation + available_resources))

        # Consume resources from environment
        consumed = self.model.environment.consume_resource(x, y, actual_consumption_rate)
        self.total_consumed += consumed

        # Convert consumed resources to biomass (with efficiency)
        conversion_efficiency = 0.7  # 70% of consumed resources become biomass
        biomass_gain = consumed * conversion_efficiency

        # Pay metabolic costs
        maintenance_cost = self.metabolic_cost * self.biomass
        net_growth = biomass_gain - maintenance_cost

        # Update biomass and energy
        if net_growth > 0:
            self.biomass += net_growth
            self.energy += net_growth * 0.5  # Some energy stored
            self.growth_rate = net_growth / self.biomass  # Specific growth rate
        else:
            # Lose biomass if can't meet maintenance costs
            self.biomass += net_growth  # net_growth is negative
            self.energy += net_growth
            self.growth_rate = net_growth / self.biomass

        # Ensure minimum biomass
        self.biomass = max(0.1, self.biomass)
        self.energy = max(0.0, self.energy)

    def attempt_division(self):
        """
        Attempt cell division when biomass exceeds threshold.
        Creates a new bacterium with potentially mutated parameters.
        """
        if self.biomass >= self.max_biomass and not self.is_dividing:
            self.is_dividing = True

            # Find an empty neighboring cell for the daughter
            neighbors = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=False, radius=1
            )

            empty_neighbors = [pos for pos in neighbors
                             if len(self.model.grid.get_cell_list_contents([pos])) == 0]

            if empty_neighbors:
                # Choose random empty neighbor
                daughter_pos = self.random.choice(empty_neighbors)

                # Split biomass between mother and daughter
                mother_biomass = self.biomass / 2
                daughter_biomass = self.biomass / 2

                # Create daughter with potential mutations
                daughter_params = self.mutate_parameters()

                daughter = BacteriumAgent(
                    model=self.model,
                    initial_biomass=daughter_biomass,
                    max_biomass=daughter_params['max_biomass'],
                    consumption_rate=daughter_params['consumption_rate'],
                    metabolic_cost=daughter_params['metabolic_cost'],
                    mutation_rate=self.mutation_rate
                )

                # Set daughter generation and position
                daughter.generation = self.generation + 1
                daughter.age = 0

                # Place daughter in environment
                self.model.grid.place_agent(daughter, daughter_pos)

                # Update mother
                self.biomass = mother_biomass
                self.energy = mother_biomass * 0.5
                self.division_count += 1
                self.age = 0  # Reset age after division
                self.is_dividing = False

                # Track division in model
                self.model.total_divisions += 1

                return True

        return False

    def mutate_parameters(self):
        """
        Generate mutated parameters for daughter cell.
        Small random variations in key parameters.
        """
        params = {}

        if random.random() < self.mutation_rate:
            # Mutate max_biomass (±10%)
            factor = 1 + random.gauss(0, 0.1)
            params['max_biomass'] = max(1.5, min(3.0, self.max_biomass * factor))
        else:
            params['max_biomass'] = self.max_biomass

        if random.random() < self.mutation_rate:
            # Mutate consumption_rate (±15%)
            factor = 1 + random.gauss(0, 0.15)
            params['consumption_rate'] = max(0.1, min(2.0, self.consumption_rate * factor))
        else:
            params['consumption_rate'] = self.consumption_rate

        if random.random() < self.mutation_rate:
            # Mutate metabolic_cost (±5%)
            factor = 1 + random.gauss(0, 0.05)
            params['metabolic_cost'] = max(0.05, min(0.5, self.metabolic_cost * factor))
        else:
            params['metabolic_cost'] = self.metabolic_cost

        return params

    def check_death(self):
        """Check if bacterium should die due to starvation."""
        # Die if biomass falls too low
        if self.biomass < 0.2 or self.energy < 0:
            self.model.deaths += 1
            self.remove()
            return True
        return False

    def step(self):
        """Execute one step of the bacterium's life cycle."""
        if not self.check_death():
            self.age += 1
            self.move()
            self.consume_and_grow()
            self.attempt_division()



In [4]:
class BacterialGrowthModel(Model):
    """
    A model simulating bacterial population dynamics with:
    - Resource consumption and competition
    - Individual growth and division
    - Spatial heterogeneity
    - Population-level emergent properties
    """

    def __init__(self, width=50, height=50, initial_bacteria=20,
                 initial_resource_level=100.0, regeneration_rate=0.1,
                 carrying_capacity_factor=1.0, seed=None):
        """
        Initialize the bacterial growth model.

        Args:
            width: Grid width
            height: Grid height
            initial_bacteria: Starting number of bacteria
            initial_resource_level: Initial resource per grid cell
            regeneration_rate: Rate of resource regeneration
            carrying_capacity_factor: Multiplier for carrying capacity
            seed: Random seed for reproducibility
        """
        super().__init__(seed=seed)

        # Model parameters
        self.width = width
        self.height = height
        self.initial_bacteria = initial_bacteria
        self.carrying_capacity = int(width * height * carrying_capacity_factor * 0.1)

        # Initialize grid
        self.grid = MultiGrid(width, height, torus=True)

        # Initialize resource environment
        self.environment = ResourceEnvironment(
            width, height, initial_resource_level, regeneration_rate
        )

        # Tracking variables
        self.total_divisions = 0
        self.deaths = 0
        self.step_count = 0

        # Create initial bacterial population
        self.create_initial_population()

        # Data collection setup
        self.setup_data_collection()

    def create_initial_population(self):
        """Create and place initial bacterial population."""
        for i in range(self.initial_bacteria):
            # Create bacterium with random parameters
            bacterium = BacteriumAgent(
                model=self,
                initial_biomass=random.uniform(0.8, 1.2),
                max_biomass=random.uniform(1.8, 2.2),
                consumption_rate=random.uniform(0.4, 0.6),
                metabolic_cost=random.uniform(0.08, 0.12)
            )

            # Place in random location
            x = self.random.randrange(self.width)
            y = self.random.randrange(self.height)
            self.grid.place_agent(bacterium, (x, y))

    def setup_data_collection(self):
        """Setup data collection for model metrics."""
        self.datacollector = DataCollector(
            model_reporters={
                "Population": lambda m: len(m.agents),
                "Total Biomass": self.get_total_biomass,
                "Average Biomass": self.get_average_biomass,
                "Total Divisions": lambda m: m.total_divisions,
                "Deaths": lambda m: m.deaths,
                "Average Resources": self.get_average_resources,
                "Resource Depletion": self.get_resource_depletion,
                "Spatial Clustering": self.get_spatial_clustering,
                "Growth Rate": self.get_population_growth_rate,
                "Carrying Capacity Utilization": self.get_carrying_capacity_utilization
            },
            agent_reporters={
                "Biomass": "biomass",
                "Age": "age",
                "Generation": "generation",
                "Consumption Rate": "consumption_rate",
                "Position X": lambda a: a.pos[0],
                "Position Y": lambda a: a.pos[1]
            }
        )

    def get_total_biomass(self):
        """Calculate total biomass of all bacteria."""
        return sum(agent.biomass for agent in self.agents)

    def get_average_biomass(self):
        """Calculate average biomass per bacterium."""
        if len(self.agents) == 0:
            return 0
        return self.get_total_biomass() / len(self.agents)

    def get_average_resources(self):
        """Calculate average resource level across grid."""
        return np.mean(self.environment.resources)

    def get_resource_depletion(self):
        """Calculate percentage of resources depleted."""
        max_resources = self.environment.max_resources
        current_avg = self.get_average_resources()
        return (1 - current_avg / max_resources) * 100

    def get_spatial_clustering(self):
        """
        Calculate spatial clustering index based on nearest neighbor distances.
        Higher values indicate more clustering.
        """
        if len(self.agents) < 2:
            return 0

        positions = [agent.pos for agent in self.agents]
        distances = []

        for i, pos1 in enumerate(positions):
            min_dist = float('inf')
            for j, pos2 in enumerate(positions):
                if i != j:
                    dist = math.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
                    min_dist = min(min_dist, dist)
            distances.append(min_dist)

        return 1.0 / (np.mean(distances) + 0.01)  # Inverse of mean nearest neighbor distance

    def get_population_growth_rate(self):
        """Calculate instantaneous population growth rate."""
        if hasattr(self, 'previous_population') and self.previous_population > 0:
            current_pop = len(self.agents)
            growth_rate = (current_pop - self.previous_population) / self.previous_population
            return growth_rate
        return 0

    def get_carrying_capacity_utilization(self):
        """Calculate what fraction of carrying capacity is being used."""
        return len(self.agents) / self.carrying_capacity

    def step(self):
        """Advance the model by one step."""
        self.previous_population = len(self.agents)

        # Update environment (resource regeneration and diffusion)
        self.environment.step()

        # Activate all agents
        self.agents.shuffle_do("step")

        # Collect data
        self.datacollector.collect(self)

        self.step_count += 1

    def run_model(self, steps):
        """Run the model for a specified number of steps."""
        for _ in range(steps):
            self.step()

print("Bacterial Growth Model classes defined successfully!")



Bacterial Growth Model classes defined successfully!


In [5]:
def create_resource_heatmap(model, ax=None, title="Resource Distribution"):
    """Create a heatmap showing resource distribution."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Create heatmap of resources
    im = ax.imshow(model.environment.resources.T, cmap='YlOrRd',
                   origin='lower', aspect='equal', interpolation='nearest')

    ax.set_title(title)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')

    # Add colorbar
    plt.colorbar(im, ax=ax, label='Resource Level')

    return ax

def create_population_heatmap(model, ax=None, title="Population Density"):
    """Create a heatmap showing bacterial population density."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Create population density grid
    pop_density = np.zeros((model.width, model.height))
    for agent in model.agents:
        x, y = agent.pos
        pop_density[x, y] += 1

    # Create heatmap
    im = ax.imshow(pop_density.T, cmap='Blues', origin='lower',
                   aspect='equal', interpolation='nearest')

    ax.set_title(title)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')

    # Add colorbar
    plt.colorbar(im, ax=ax, label='Number of Bacteria')

    return ax

def create_biomass_heatmap(model, ax=None, title="Biomass Distribution"):
    """Create a heatmap showing total biomass distribution."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # Create biomass density grid
    biomass_density = np.zeros((model.width, model.height))
    for agent in model.agents:
        x, y = agent.pos
        biomass_density[x, y] += agent.biomass

    # Create heatmap
    im = ax.imshow(biomass_density.T, cmap='Greens', origin='lower',
                   aspect='equal', interpolation='nearest')

    ax.set_title(title)
    ax.set_xlabel('X Position')
    ax.set_ylabel('Y Position')

    # Add colorbar
    plt.colorbar(im, ax=ax, label='Total Biomass')

    return ax

def plot_population_dynamics(model_data, figsize=(12, 8)):
    """
    Plot comprehensive population dynamics over time.
    Shows population growth, biomass, and environmental metrics.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle('Bacterial Population Dynamics Over Time', fontsize=16)

    steps = range(len(model_data['Population']))

    # Population growth
    axes[0, 0].plot(steps, model_data['Population'], 'b-', linewidth=2, label='Population')
    axes[0, 0].set_xlabel('Time Steps')
    axes[0, 0].set_ylabel('Number of Bacteria')
    axes[0, 0].set_title('Population Growth')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    # Biomass dynamics
    axes[0, 1].plot(steps, model_data['Total Biomass'], 'g-', linewidth=2, label='Total Biomass')
    axes[0, 1].plot(steps, model_data['Average Biomass'], 'orange', linewidth=2, label='Avg Biomass')
    axes[0, 1].set_xlabel('Time Steps')
    axes[0, 1].set_ylabel('Biomass')
    axes[0, 1].set_title('Biomass Dynamics')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    # Resource depletion
    axes[1, 0].plot(steps, model_data['Average Resources'], 'r-', linewidth=2, label='Avg Resources')
    axes[1, 0].plot(steps, model_data['Resource Depletion'], 'm-', linewidth=2, label='Depletion %')
    axes[1, 0].set_xlabel('Time Steps')
    axes[1, 0].set_ylabel('Resource Level / Depletion %')
    axes[1, 0].set_title('Resource Dynamics')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    # Carrying capacity and spatial metrics
    axes[1, 1].plot(steps, model_data['Carrying Capacity Utilization'], 'purple',
                   linewidth=2, label='Carrying Capacity Use')
    axes[1, 1].plot(steps, model_data['Spatial Clustering'], 'brown',
                   linewidth=2, label='Spatial Clustering')
    axes[1, 1].set_xlabel('Time Steps')
    axes[1, 1].set_ylabel('Index Value')
    axes[1, 1].set_title('Capacity & Spatial Metrics')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    plt.tight_layout()
    return fig

def create_growth_phase_analysis(model_data):
    """
    Analyze and visualize different growth phases (lag, exponential, stationary).
    """
    population = np.array(model_data['Population'])
    steps = np.array(range(len(population)))

    # Calculate growth rate (derivative of log population)
    log_pop = np.log(population + 1)  # Add 1 to avoid log(0)
    growth_rates = np.gradient(log_pop)

    # Identify growth phases
    # Lag phase: low growth rate at beginning
    # Exponential phase: high, stable growth rate
    # Stationary phase: declining growth rate

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Growth Phase Analysis', fontsize=16)

    # Population on linear scale
    axes[0, 0].plot(steps, population, 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Time Steps')
    axes[0, 0].set_ylabel('Population')
    axes[0, 0].set_title('Population Growth (Linear Scale)')
    axes[0, 0].grid(True, alpha=0.3)

    # Population on log scale
    axes[0, 1].semilogy(steps, population, 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Time Steps')
    axes[0, 1].set_ylabel('Population (log scale)')
    axes[0, 1].set_title('Population Growth (Log Scale)')
    axes[0, 1].grid(True, alpha=0.3)

    # Growth rate over time
    axes[1, 0].plot(steps, growth_rates, 'r-', linewidth=2)
    axes[1, 0].set_xlabel('Time Steps')
    axes[1, 0].set_ylabel('Instantaneous Growth Rate')
    axes[1, 0].set_title('Growth Rate Over Time')
    axes[1, 0].grid(True, alpha=0.3)

    # Phase diagram: growth rate vs population
    axes[1, 1].scatter(population, growth_rates, c=steps, cmap='viridis', alpha=0.6)
    axes[1, 1].set_xlabel('Population')
    axes[1, 1].set_ylabel('Growth Rate')
    axes[1, 1].set_title('Growth Rate vs Population (colored by time)')
    axes[1, 1].grid(True, alpha=0.3)

    # Add colorbar for time
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=0, vmax=len(steps)))
    sm.set_array([])
    plt.colorbar(sm, ax=axes[1, 1], label='Time Step')

    plt.tight_layout()
    return fig

def analyze_agent_heterogeneity(model):
    """
    Analyze heterogeneity in bacterial population characteristics.
    """
    if len(model.agents) == 0:
        print("No agents to analyze.")
        return None

    # Collect agent data
    biomasses = [agent.biomass for agent in model.agents]
    ages = [agent.age for agent in model.agents]
    generations = [agent.generation for agent in model.agents]
    consumption_rates = [agent.consumption_rate for agent in model.agents]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Bacterial Population Heterogeneity', fontsize=16)

    # Biomass distribution
    axes[0, 0].hist(biomasses, bins=20, alpha=0.7, color='green', edgecolor='black')
    axes[0, 0].set_xlabel('Biomass')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].set_title('Biomass Distribution')
    axes[0, 0].grid(True, alpha=0.3)

    # Age distribution
    axes[0, 1].hist(ages, bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[0, 1].set_xlabel('Age (steps)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Age Distribution')
    axes[0, 1].grid(True, alpha=0.3)

    # Generation distribution
    axes[1, 0].hist(generations, bins=max(1, max(generations)), alpha=0.7, color='red', edgecolor='black')
    axes[1, 0].set_xlabel('Generation')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Generation Distribution')
    axes[1, 0].grid(True, alpha=0.3)

    # Consumption rate vs biomass
    axes[1, 1].scatter(consumption_rates, biomasses, alpha=0.6, c=ages, cmap='plasma')
    axes[1, 1].set_xlabel('Consumption Rate')
    axes[1, 1].set_ylabel('Biomass')
    axes[1, 1].set_title('Consumption Rate vs Biomass (colored by age)')
    axes[1, 1].grid(True, alpha=0.3)

    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(vmin=min(ages), vmax=max(ages)))
    sm.set_array([])
    plt.colorbar(sm, ax=axes[1, 1], label='Age')

    plt.tight_layout()
    return fig

def create_trajectory_animation(model_history, save_path=None, interval=200):
    """
    Create an animated visualization of the complete simulation trajectory.
    Shows bacterial movement, resource depletion, and population dynamics.
    """
    if not model_history:
        print("No model history provided for animation.")
        return None

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('Bacterial Growth Simulation Trajectory', fontsize=16)

    # Initialize plots
    resource_im = axes[0].imshow(model_history[0].environment.resources.T,
                                cmap='YlOrRd', origin='lower', aspect='equal')
    axes[0].set_title('Resource Distribution')
    axes[0].set_xlabel('X Position')
    axes[0].set_ylabel('Y Position')
    plt.colorbar(resource_im, ax=axes[0], label='Resource Level')

    # Population scatter plot
    scat = axes[1].scatter([], [], c=[], s=[], cmap='viridis', alpha=0.7)
    axes[1].set_xlim(0, model_history[0].width)
    axes[1].set_ylim(0, model_history[0].height)
    axes[1].set_title('Bacterial Population')
    axes[1].set_xlabel('X Position')
    axes[1].set_ylabel('Y Position')

    # Population dynamics plot
    line1, = axes[2].plot([], [], 'b-', linewidth=2, label='Population')
    line2, = axes[2].plot([], [], 'g-', linewidth=2, label='Total Biomass')
    axes[2].set_xlabel('Time Steps')
    axes[2].set_ylabel('Count / Biomass')
    axes[2].set_title('Population Dynamics')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    # Data for animation
    populations = []
    biomasses = []

    def animate(frame):
        model = model_history[frame]

        # Update resource heatmap
        resource_im.set_array(model.environment.resources.T)
        resource_im.set_clim(vmin=0, vmax=model.environment.max_resources)

        # Update bacterial positions and properties
        if len(model.agents) > 0:
            positions = np.array([agent.pos for agent in model.agents])
            biomasses_frame = np.array([agent.biomass for agent in model.agents])
            ages = np.array([agent.age for agent in model.agents])

            # Size based on biomass, color based on age
            sizes = biomasses_frame * 50  # Scale for visibility
            colors = ages

            # Update scatter plot
            scat.set_offsets(positions)
            scat.set_sizes(sizes)
            scat.set_array(colors)
        else:
            scat.set_offsets(np.empty((0, 2)))
            scat.set_sizes([])
            scat.set_array([])

        # Update population dynamics
        populations.append(len(model.agents))
        biomasses.append(sum(agent.biomass for agent in model.agents))

        steps = range(len(populations))
        line1.set_data(steps, populations)
        line2.set_data(steps, biomasses)

        # Update axes limits
        if populations:
            axes[2].set_xlim(0, max(10, len(populations)))
            axes[2].set_ylim(0, max(10, max(max(populations), max(biomasses))))

        # Update title with current step
        fig.suptitle(f'Bacterial Growth Simulation - Step {frame}', fontsize=16)

        return resource_im, scat, line1, line2

    # Create animation
    anim = animation.FuncAnimation(fig, animate, frames=len(model_history),
                                 interval=interval, blit=False, repeat=True)

    if save_path:
        anim.save(save_path, writer='pillow', fps=5)
        print(f"Animation saved to {save_path}")

    plt.tight_layout()
    return anim



In [6]:
def run_growth_experiment(width=30, height=30, initial_bacteria=10, steps=100,
                         resource_level=100, regeneration_rate=0.1,
                         carrying_capacity_factor=1.0, seed=42):
    """
    Run a complete bacterial growth experiment with visualization.
    """
    print(f"Starting bacterial growth experiment...")
    print(f"Grid: {width}x{height}, Initial bacteria: {initial_bacteria}")
    print(f"Resource level: {resource_level}, Regeneration: {regeneration_rate}")
    print(f"Steps: {steps}, Seed: {seed}")
    print("-" * 50)

    # Create and run model
    model = BacterialGrowthModel(
        width=width, height=height, initial_bacteria=initial_bacteria,
        initial_resource_level=resource_level, regeneration_rate=regeneration_rate,
        carrying_capacity_factor=carrying_capacity_factor, seed=seed
    )

    # Store model states for animation
    model_history = []

    # Run simulation with progress tracking
    for step in range(steps):
        if step % 20 == 0:
            print(f"Step {step}: Population = {len(model.agents)}, "
                  f"Avg Resources = {model.get_average_resources():.1f}")

        # Store model state (deep copy for animation)
        if step % 5 == 0:  # Store every 5th step to save memory
            import copy
            model_copy = copy.deepcopy(model)
            model_history.append(model_copy)

        model.step()

        # Check for extinction
        if len(model.agents) == 0:
            print(f"Population extinct at step {step}")
            break

    # Get final data
    model_data = model.datacollector.get_model_vars_dataframe()
    agent_data = model.datacollector.get_agent_vars_dataframe()

    print(f"Experiment completed!")
    print(f"Final population: {len(model.agents)}")
    print(f"Total divisions: {model.total_divisions}")
    print(f"Total deaths: {model.deaths}")
    print(f"Final avg resources: {model.get_average_resources():.1f}")

    return model, model_data, agent_data, model_history

def create_comprehensive_analysis(model, model_data, agent_data):
    """
    Create a comprehensive analysis dashboard with all visualizations.
    """
    print("Creating comprehensive analysis dashboard...")

    # Create figure with subplots
    fig = plt.figure(figsize=(20, 16))

    # Population dynamics (top row)
    plt.subplot(3, 4, (1, 2))
    steps = range(len(model_data['Population']))
    plt.plot(steps, model_data['Population'], 'b-', linewidth=2, label='Population')
    plt.xlabel('Time Steps')
    plt.ylabel('Number of Bacteria')
    plt.title('Population Growth Over Time')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(3, 4, (3, 4))
    plt.plot(steps, model_data['Total Biomass'], 'g-', linewidth=2, label='Total Biomass')
    plt.plot(steps, model_data['Average Biomass'], 'orange', linewidth=2, label='Avg Biomass')
    plt.xlabel('Time Steps')
    plt.ylabel('Biomass')
    plt.title('Biomass Dynamics')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Spatial distribution (middle row)
    plt.subplot(3, 4, 5)
    create_resource_heatmap(model, plt.gca(), "Resources")

    plt.subplot(3, 4, 6)
    create_population_heatmap(model, plt.gca(), "Population Density")

    plt.subplot(3, 4, 7)
    create_biomass_heatmap(model, plt.gca(), "Biomass Distribution")

    plt.subplot(3, 4, 8)
    plt.plot(steps, model_data['Average Resources'], 'r-', linewidth=2, label='Avg Resources')
    plt.plot(steps, model_data['Resource Depletion'], 'm-', linewidth=2, label='Depletion %')
    plt.xlabel('Time Steps')
    plt.ylabel('Resource Level / %')
    plt.title('Resource Dynamics')
    plt.grid(True, alpha=0.3)
    plt.legend()

    # Growth analysis (bottom row)
    plt.subplot(3, 4, 9)
    if len(model.agents) > 0:
        biomasses = [agent.biomass for agent in model.agents]
        plt.hist(biomasses, bins=15, alpha=0.7, color='green', edgecolor='black')
    plt.xlabel('Biomass')
    plt.ylabel('Frequency')
    plt.title('Current Biomass Distribution')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 4, 10)
    plt.plot(steps, model_data['Carrying Capacity Utilization'], 'purple',
             linewidth=2, label='Carrying Capacity')
    plt.plot(steps, model_data['Spatial Clustering'], 'brown',
             linewidth=2, label='Spatial Clustering')
    plt.xlabel('Time Steps')
    plt.ylabel('Index Value')
    plt.title('Capacity & Spatial Metrics')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.subplot(3, 4, 11)
    growth_rates = np.gradient(np.log(np.array(model_data['Population']) + 1))
    plt.plot(steps, growth_rates, 'red', linewidth=2)
    plt.xlabel('Time Steps')
    plt.ylabel('Growth Rate')
    plt.title('Instantaneous Growth Rate')
    plt.grid(True, alpha=0.3)

    plt.subplot(3, 4, 12)
    plt.plot(steps, model_data['Total Divisions'], 'blue', linewidth=2, label='Divisions')
    plt.plot(steps, model_data['Deaths'], 'red', linewidth=2, label='Deaths')
    plt.xlabel('Time Steps')
    plt.ylabel('Count')
    plt.title('Divisions vs Deaths')
    plt.grid(True, alpha=0.3)
    plt.legend()

    plt.suptitle('Comprehensive Bacterial Growth Analysis', fontsize=16)
    plt.tight_layout()

    return fig

print("Visualization and analysis functions created successfully!")

Visualization and analysis functions created successfully!


In [7]:
def interactive_experiment():
    """
    Create interactive widgets for experimenting with model parameters.
    """
    print("Creating interactive experiment interface...")

    @interact(
        width=IntSlider(min=20, max=80, step=10, value=20, description='Grid Width'),
        height=IntSlider(min=20, max=80, step=10, value=20, description='Grid Height'),
        initial_bacteria=IntSlider(min=5, max=100, step=5, value=10, description='Initial Bacteria'),
        steps=IntSlider(min=50, max=300, step=50, value=200, description='Simulation Steps'),
        resource_level=FloatSlider(min=50, max=200, step=25, value=100, description='Resource Level'),
        regeneration_rate=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.05, description='Regeneration Rate'),
        carrying_capacity_factor=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0, description='Carrying Capacity'),
        seed=IntSlider(min=1, max=100, step=1, value=42, description='Random Seed')
    )
    def run_interactive_experiment(width, height, initial_bacteria, steps,
                                 resource_level, regeneration_rate,
                                 carrying_capacity_factor, seed):
        """Run experiment with interactive parameters."""
        clear_output(wait=True)

        # Run the experiment
        model, model_data, agent_data, model_history = run_growth_experiment(
            width=width, height=height, initial_bacteria=initial_bacteria,
            steps=steps, resource_level=resource_level,
            regeneration_rate=regeneration_rate,
            carrying_capacity_factor=carrying_capacity_factor, seed=seed
        )

        # Create comprehensive analysis
        fig = create_comprehensive_analysis(model, model_data, agent_data)
        plt.show()

        return model, model_data, agent_data



## Interactive Experiments

Use the interactive interface to explore how different parameters affect bacterial growth:

In [8]:
# Run interactive experiment with parameter sliders
interactive_experiment()

Creating interactive experiment interface...


interactive(children=(IntSlider(value=20, description='Grid Width', max=80, min=20, step=10), IntSlider(value=…

## Key Concepts Demonstrated

### 🦠 Individual Agent Behaviors
- **Movement**: Random walk with chemotaxis toward resources
- **Growth**: Biomass accumulation through resource consumption
- **Division**: Cell division when reaching critical biomass
- **Death**: Starvation when resources are insufficient
- **Mutation**: Parameter variation in daughter cells

### 🌍 Environmental Dynamics
- **Resource depletion**: Consumption reduces local nutrients
- **Resource regeneration**: Slow recovery of depleted areas
- **Resource diffusion**: Spreading of nutrients between cells
- **Spatial heterogeneity**: Uneven resource distribution

### 📊 Population-Level Patterns
- **Exponential growth**: Rapid increase in resource-rich conditions
- **Logistic growth**: S-shaped curve with carrying capacity
- **Resource competition**: Competition for limited nutrients
- **Spatial clustering**: Aggregation around resource patches
- **Population cycles**: Boom-bust dynamics

## Discussion Questions

1. **Growth Patterns**: How do different resource conditions affect the shape of the growth curve? When do you observe exponential vs logistic growth?

2. **Spatial Dynamics**: Why do bacteria form clusters? How does resource depletion create spatial heterogeneity?

3. **Carrying Capacity**: What factors determine the maximum population size? How does it relate to resource availability?

4. **Competition**: How do bacteria compete for resources? What strategies emerge from individual behaviors?

5. **Real-World Applications**: How might these patterns apply to real bacterial communities in nature or laboratory settings?

## Extensions and Future Work

This model can be extended to explore:
- **Multiple bacterial species** with different characteristics
- **Antibiotic resistance** and treatment strategies
- **Biofilm formation** and community structure
- **Environmental gradients** and migration patterns
- **Metabolic interactions** and cooperation

---

*This notebook demonstrates advanced agent-based modeling concepts using Mesa 3.X, building upon basic movement and chemotaxis to explore complex population dynamics and emergent behaviors in bacterial communities.*